# Vector Spaces of Hermitian Matrices and Pauli Basis

This notebook demonstrates numerical linear algebra in Python through Hermitian matrix spaces, basis transformations, and operator representation. It is designed to show clear programming structure, validated computation, and reproducible results.

We construct:
- the Pauli basis for $2\times 2$ Hermitian matrices,
- a custom basis transformation and its Kronecker extension,
- the matrix representation of the adjoint operator $\mathrm{ad}(H)$.

Each section includes both computation and validation.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# Set dimensions for the matrix space
dim = 2

### Pauli Basis Linear Independence Validation

In [ ]:
sigma_0 = np.array([[1, 0], [0, 1]], dtype=complex)
sigma_1 = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_2 = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_3 = np.array([[1, 0], [0, -1]], dtype=complex)

pauli_basis = [sigma_0, sigma_1, sigma_2, sigma_3]
B2 = np.array([M.ravel() for M in pauli_basis]).T

det_B2 = np.linalg.det(B2)
print(f"Determinant of the Pauli basis matrix: {det_B2:.4g}")
if np.abs(det_B2) > 0:
    print("Verification passed: The Pauli matrices form a valid basis for V^2.")
else:
    print("Verification failed: The Pauli matrices are linearly dependent.")

### 1. Basis Transformation in $V^2$

We can define an alternative arbitrary basis $\tilde{\mathcal{B}}_2$ for the $V^2$ space. Using `numpy.linalg.solve`, we compute the transition matrix $C_2$ that represents the coordinates of this new basis with respect to our standard Pauli basis $\mathcal{B}_2$.

In [ ]:
A0 = np.array([[0.8485+0j, 0.4215-0.1026j],
               [0.4215+0.1026j, 1.704+0j]])

A1 = np.array([[0.89+0j, -0.5197+1.1544j],
               [-0.5197-1.1544j, -0.0471+0j]])

A2 = np.array([[0.6704+0j, 1.1544+0.2435j],
               [1.1544-0.2435j, -0.8755+0j]])

A3 = np.array([[-1.4278+0j, 0.4685+0.773j],
               [0.4685-0.773j, 0.5722+0j]])

B2_tilde = np.array([A0.ravel(), A1.ravel(), A2.ravel(), A3.ravel()]).T

det_B2_tilde = np.linalg.det(B2_tilde)
print(f"Determinant of transformed basis matrix: {det_B2_tilde:.4g}")
if np.abs(det_B2_tilde) > 0:
    print("Verification passed: The custom matrices form a basis for V^2.")
else:
    print("Verification failed: The custom matrices do not form a basis for V^2.")

In [ ]:
B2 = np.array([sigma_0.ravel(), sigma_1.ravel(), sigma_2.ravel(), sigma_3.ravel()]).T

C2 = np.linalg.solve(B2, B2_tilde)
print("Transition matrix C2 from the Pauli basis to the custom basis:")
print(C2)

reconstruction_ok = np.allclose(B2 @ C2, B2_tilde, atol=1e-10)
print("Reconstruction check (B2 @ C2 == B2_tilde):", reconstruction_ok)
print(f"The real part of the transition matrix element C2(3,3) is: {np.real(C2[2, 2]):.4f}")

### 2. Transition Matrices in Higher Dimensions

By applying the Kronecker product to our basis transformations, we can compute the transition matrix $C_4$ for the 16-dimensional space of $4\times 4$ matrices. We will programmatically verify the algebraic property $C_4 = C_2 \otimes C_2$.

In [ ]:
pauli_basis = [sigma_0, sigma_1, sigma_2, sigma_3]
B4 = np.array([np.kron(p, q).ravel() for p in pauli_basis for q in pauli_basis]).T

det_B4 = np.linalg.det(B4)
print(f"Determinant of the Kronecker-product basis matrix: {det_B4:.4g}")
if np.abs(det_B4) > 0:
    print("Verification passed: B4 is a valid basis for 4x4 matrices.")
else:
    print("Verification failed: B4 is not a valid basis for 4x4 matrices.")

In [ ]:
B4_tilde = np.kron(B2_tilde, B2_tilde)

C4 = np.linalg.solve(B4, B4_tilde)
C4_expected = np.kron(C2, C2)

print("Transition matrix C4 computed from the Kronecker basis:")
print(C4)
print("\nC4 equal to C2 ⊗ C2:", np.allclose(C4, C4_expected, atol=1e-10))
print(f"Sample element C4[12,12]: {np.real(C4[12,12]):.4f}")

#### Algebraic Verification: $C_4=C_2\otimes C_2$

In [ ]:
C4_expected = np.kron(C2, C2)
if np.allclose(C4, C4_expected, atol=1e-10):
    print("Algebraic property confirmed: C4 = C2 ⊗ C2")
else:
    print("The matrices do not match.")
    print("Frobenius norm of the difference:", np.linalg.norm(C4 - C4_expected))

### 3. Matrix Representation of the Adjoint Endomorphism

Let $H$ be an arbitrary $4 \times 4$ Hermitian matrix. We analyze the endomorphism defined by the commutator:
$$\mathrm{ad}(H): V^4 \to V^4; \quad \rho \mapsto -i [H,\rho] = -i(H\rho - \rho H)$$

The following script computes the exact matrix representation of this operator with respect to our constructed basis.

In [ ]:
H = np.array([
    [0.1571+0j, 0+0j, 0+0j, -0.3459+0j],
    [0+0j, -0.1571+0j, 0.5961+0j, 0+0j],
    [0+0j, 0.5961+0j, -0.1571+0j, 0+0j],
    [-0.3459+0j, 0+0j, 0+0j, 0.1571+0j]
], dtype=complex)

# Commutator operator -i[H,A]
def adH(A):
    return -1j * (H @ A - A @ H)

# Generate a normalized basis of Hermitian 4x4 matrices (dim=16)
def generate_hermitian_basis():
    basis = []
    for i in range(4):
        for j in range(i, 4):
            if i == j:
                mat = np.zeros((4, 4), dtype=complex)
                mat[i, j] = 1
                basis.append(mat)
            else:
                mat_real = np.zeros((4, 4), dtype=complex)
                mat_real[i, j] = mat_real[j, i] = 1
                basis.append(mat_real)
                mat_imag = np.zeros((4, 4), dtype=complex)
                mat_imag[i, j] = 1j
                mat_imag[j, i] = -1j
                basis.append(mat_imag)
    return [M / np.linalg.norm(M) for M in basis]

B4 = generate_hermitian_basis()

# Check basis orthonormality under the Frobenius inner product
inner = np.array([[np.trace(np.conj(Bi.T) @ Bj) for Bj in B4] for Bi in B4])
print("Orthonormality check for the Hermitian basis:", np.allclose(inner, np.eye(16), atol=1e-10))

# Calculate the matrix representation of the ad(H) operator in the B4 basis
C = np.zeros((16, 16), dtype=complex)
for i, Bi in enumerate(B4):
    X = adH(Bi)
    for j, Bj in enumerate(B4):
        C[j, i] = np.trace(np.conj(Bj.T) @ X)

# Display specific matrix elements (real part)
C_real = np.real(C)

print(f"Value at index (2,13): {C_real[1,12]:.4f}")
print(f"Value at index (12,2): {C_real[11,1]:.4f}")

### Summary

This notebook demonstrates:
- construction and validation of Pauli and custom matrix bases,
- basis change via transition matrices,
- Kronecker structure for higher-dimensional matrix spaces,
- orthonormal Hermitian basis generation and operator representation for $\mathrm{ad}(H)$.

The code is organized for reproducibility and numerical verification.